# Homework: Wikipedia Agent

Build an agent that can search Wikipedia and fetch page content to answer questions.

## Setup: imports

In [1]:
import requests
from typing import Any, Dict, List

## Setup: Wikipedia tools

A `WikipediaTools` class with `search` and `get_page`. It isn't part of a specific question yet; it's here for when we build the agent.

In [2]:
class WikipediaTools:
    """
    Provides search and page retrieval utilities over the Wikipedia API.
    """

    headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search Wikipedia for pages related to a query.

        Args:
            query (str): The search query.

        Returns:
            List[Dict[str, Any]]: Search results, each with a page title and snippet.
        """
        response = requests.get(
            "https://en.wikipedia.org/w/api.php",
            params={
                "action": "query",
                "format": "json",
                "list": "search",
                "srsearch": query,
            },
            headers=self.headers,
        )
        response.raise_for_status()
        return response.json()["query"]["search"]

    def get_page(self, title: str) -> str:
        """
        Fetch the raw wikitext content of a Wikipedia page.

        Args:
            title (str): The exact title of the Wikipedia page.

        Returns:
            str: The raw content of the page.
        """
        response = requests.get(
            "https://en.wikipedia.org/w/index.php",
            params={"title": title, "action": "raw"},
            headers=self.headers,
        )
        response.raise_for_status()
        return response.text

In [3]:
wiki_tools = WikipediaTools()

results = wiki_tools.search("lesser capybara")
[r["title"] for r in results]

['Lesser capybara',
 'Capybara',
 'Hydrochoerus',
 'Capybara (disambiguation)',
 'Caviidae',
 'Hydrochoerus hesperotiganites',
 'Kerodon',
 'List of largest rodents',
 'Cavia',
 'Dolichotis']

In [4]:
page = wiki_tools.get_page("Capybara")
len(page), page[:500]

(44767,
 '{{Short description|Largest species of rodents}}\n{{Other uses}}\n{{Good article}}\n{{pp|small=yes}}\n{{Use dmy dates|date=July 2022}}\n{{Speciesbox\n| status = LC\n| status_system = IUCN3.1\n| status_ref = <ref name="iucn">{{cite iucn |author=Reid, F. |year=2016 |title=\'\'Hydrochoerus hydrochaeris\'\' |article-number=e.T10300A22190005 |doi=10.2305/IUCN.UK.2016-2.RLTS.T10300A22190005.en}}</ref>\n| fossil_range = {{Fossilrange|Late Pleistocene|Present}}\n| image = Capybaracropped.jpg\n| image_alt = A capybara ')

## Question 1. Define the Search Tool

Create a `search_wikipedia` function that takes a query, calls the Wikipedia Search API, and returns the results. Test it with "capybara".

**How many total results did the search return?** Count the entries in the `query.search` array.

**Answer: 10**

In [5]:
headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}


def search_wikipedia(query: str) -> Dict[str, Any]:
    """
    Search Wikipedia for pages related to a query.

    Args:
        query (str): The search query.

    Returns:
        Dict[str, Any]: The search response, with "searchinfo" (total hits)
        and "search" (the list of matching pages).
    """
    response = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "query",
            "format": "json",
            "list": "search",
            "srsearch": query,
        },
        headers=headers,
    )
    response.raise_for_status()
    return response.json()["query"]

In [6]:
results = search_wikipedia("capybara")

# Q1 answer: number of entries in query.search
len(results["search"])

10

## Question 2. Analyzing Search Results

**How many of the results contain the word "capybara" (case-insensitive) in their title?**

**Answer: 6**

In [7]:
titles = [r["title"] for r in results["search"]]
matching = [t for t in titles if "capybara" in t.lower()]

# Q2 answer
matching, len(matching)

(['Capybara',
  'Lesser capybara',
  'Capybara Games',
  'Capybara (disambiguation)',
  'Capybara Party',
  'Capybara (software)'],
 6)

## Question 3. Define the Get Page Tool

Create a `get_page` function that takes a page title, calls the Wikipedia Get Page API, and returns the page content as a string. Test it with "Capybara".

**How many characters are in the page content?**

**Answer: ~45,000**

In [8]:
def get_page(title: str) -> str:
    """
    Fetch the raw wikitext content of a Wikipedia page.

    Args:
        title (str): The exact title of the Wikipedia page.

    Returns:
        str: The raw content of the page.
    """
    response = requests.get(
        "https://en.wikipedia.org/w/index.php",
        params={"title": title, "action": "raw"},
        headers=headers,
    )
    response.raise_for_status()
    return response.text

In [9]:
content = get_page("Capybara")

# Q3 answer
len(content)

44767

## Question 4. Agent Setup

Create an agent with access to both `search_wikipedia` and `get_page`.

**Answer:**
- Framework: PydanticAI
- LLM provider: OpenAI (`gpt-4o-mini`)

In [10]:
from pydantic_ai import Agent

wiki_instructions = """
You're a Wikipedia research assistant.

Answer the user's question using Wikipedia:
1) Use search_wikipedia to find pages relevant to the question.
2) Use get_page to read the most relevant pages. Fetch more than one if needed.
3) Answer based only on the page content, and cite the page titles you used.
"""

wiki_agent = Agent(
    name="wikipedia",
    model="openai:gpt-4o-mini",
    instructions=wiki_instructions,
    tools=[search_wikipedia, get_page],
)

In [11]:
# Q4 test run
result = await wiki_agent.run("Where do capybaras live?")
print(result.output)

Capybaras, the largest living rodents, are native to all countries in South America except Chile. They inhabit a variety of wetland environments, including savannas, marshes, and tropical rainforests, typically living near bodies of freshwater such as rivers, lakes, ponds, and swamps. This semi-aquatic lifestyle is crucial for their feeding and social behavior, as they feed on grasses and aquatic plants. In the wild, capybaras can be found at elevations up to 1,500 meters, although they primarily thrive in lowland areas.

For more details, you can refer to the "Capybara" Wikipedia page.


## Question 5. Testing Your Agent - Single Page

Ask the agent: "What is this page about? https://en.wikipedia.org/wiki/Capybara"

**What is the main topic of the Capybara page according to your agent?**

**Answer:** The page is about the capybara (*Hydrochoerus hydrochaeris*), the largest living rodent. It's a semi-aquatic herbivore native to South America. The page covers its physical traits, social behavior, diet, reproduction, conservation status and cultural significance.

In [12]:
from pydantic_ai.messages import ToolCallPart

q5_result = await wiki_agent.run("What is this page about? https://en.wikipedia.org/wiki/Capybara")

# Check which tools the agent called
for message in q5_result.all_messages():
    for part in message.parts:
        if isinstance(part, ToolCallPart):
            print("Tool call:", part.tool_name, part.args)

# Q5 answer
print()
print(q5_result.output)

Tool call: get_page {"title":"Capybara"}

The Wikipedia page on the **Capybara** provides comprehensive information about this species, which is the largest rodent in the world. Here are some key points:

1. **Scientific Classification**: The capybara's scientific name is *Hydrochoerus hydrochaeris*. It belongs to the subfamily Hydrochoerinae within the family Caviidae, making it closely related to guinea pigs and rock cavies.

2. **Habitat**: Capybaras are native to South America and inhabit savannas and dense forests near water sources, requiring freshwater bodies like rivers and lakes. They are semi-aquatic and are often found in groups.

3. **Diet**: These animals are mainly herbivores, feeding on grasses and aquatic plants. They are selective grazers and exhibit coprophagy, meaning they consume their own feces to maximize nutrient absorption.

4. **Social Structure**: Capybaras are social animals, often living in groups ranging from 10 to over 100 individuals. Their social interac

## Question 6. Testing Your Agent - Search Then Fetch

Ask the agent: "What are the main threats to capybara populations?"

**How many total tool calls (search + fetch combined) did your agent make?**

**Answer: 5 tool calls**

Order:
1. `search_wikipedia("threats to capybara populations")`
2. `get_page("Rodent")`
3. `search_wikipedia("capybara conservation threats")`
4. `get_page("Pantanal")`
5. `get_page("Jaguar")`

Final answer (verbatim from the agent):

> Capybara populations face several significant threats, primarily linked to habitat destruction and hunting. Here are the main threats:
>
> 1. **Habitat Loss**: Deforestation, agricultural expansion, and urbanization have led to significant habitat reduction for capybaras. As wetlands and riverbanks are converted for agricultural use or urban development, capybaras lose their natural habitats, which are crucial for their survival.
>
> 2. **Hunting and Poaching**: Capybaras are often hunted for their meat and skin, which are considered valuable in many cultures. They are also targeted due to human-wildlife conflicts, as ranchers may view them as pests.
>
> 3. **Human-Wildlife Conflict**: As agricultural land expands, capybaras can become targets for farmers when they encroach on crops and livestock areas, leading to retaliatory killings.
>
> 4. **Disease and Predation**: Increased interactions with domestic animals can expose wild capybara populations to diseases. Furthermore, they are prey to various predators, including jaguars and large cats, which can impact local populations, especially where habitats overlap with human development.
>
> 5. **Climate Change**: Changes in rainfall patterns and temperature due to climate change can alter their wetland habitats, further stressing capybara populations.
>
> These threats are compounded by the ecological degradation of their environments, which reduces their food sources and breeding grounds, thus impacting their overall population viability (Sources: "Pantanal," "Jaguar").

In [13]:
q6_result = await wiki_agent.run("What are the main threats to capybara populations?")

# Tool calls in order
tool_calls = [
    part
    for message in q6_result.all_messages()
    for part in message.parts
    if isinstance(part, ToolCallPart)
]
for i, call in enumerate(tool_calls, 1):
    print(f"{i}. {call.tool_name} {call.args}")

# Q6 answer
print()
print("Total tool calls:", len(tool_calls))
print()
print(q6_result.output)

1. search_wikipedia {"query":"threats to capybara populations"}
2. get_page {"title":"Rodent"}
3. search_wikipedia {"query":"capybara conservation threats"}
4. get_page {"title":"Pantanal"}
5. get_page {"title":"Jaguar"}

Total tool calls: 5

Capybara populations face several significant threats, primarily linked to habitat destruction and hunting. Here are the main threats:

1. **Habitat Loss**: Deforestation, agricultural expansion, and urbanization have led to significant habitat reduction for capybaras. As wetlands and riverbanks are converted for agricultural use or urban development, capybaras lose their natural habitats, which are crucial for their survival.

2. **Hunting and Poaching**: Capybaras are often hunted for their meat and skin, which are considered valuable in many cultures. They are also targeted due to human-wildlife conflicts, as ranchers may view them as pests.

3. **Human-Wildlife Conflict**: As agricultural land expands, capybaras can become targets for farmers 